# AI Product Price Estimator – QLoRA + LLaMA 3.1 8B

Predict product prices from textual descriptions using a fine-tuned LLaMA model google colab notebook.

## Features
- Predict product prices from product descriptions (electronics, apparel, home goods, etc.)
- Top-K weighted token prediction for probabilistic outputs
- Evaluates performance with Dollar Error, RMSLE, and Hit Rate
- 4-bit quantized model for efficient VRAM usage
- Chain-of-Thought reasoning for interpretable outputs

In [ ]:
import sys, os, re, math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, CUDA: {torch.version.cuda}")

In [ ]:
!pip install -q --upgrade transformers==4.48.3 accelerate==1.3.0 datasets==3.2.0
!pip install -q --upgrade peft==0.14.0 trl==0.14.0 bitsandbytes==0.46.0
!pip install -q --upgrade scikit-learn matplotlib scipy
!pip install -q --upgrade "huggingface_hub<1.0,>=0.24.0"

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
from datasets import load_dataset

HF_DATASET = "ed-donner/pricer-data"  # e-commerce product dataset
data_bundle = load_dataset(HF_DATASET)
train_split = data_bundle['train']
test_split = data_bundle['test']

print(f"Training examples: {len(train_split)}, Test examples: {len(test_split)}")
print("Sample example:", test_split[0])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

LLM_BASE = "meta-llama/Meta-Llama-3.1-8B"

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tok = AutoTokenizer.from_pretrained(LLM_BASE, trust_remote_code=True)
tok.pad_token = tok.eos_token
tok.padding_side = "right"

core_model = AutoModelForCausalLM.from_pretrained(
    LLM_BASE,
    quantization_config=quantization,
    device_map="auto"
)
core_model.generation_config.pad_token_id = tok.pad_token_id

In [ ]:
ADAPTER_REPO = "ed-donner/pricer-2024-09-13_13.04.39"
ADAPTER_REV = "e8d637df551603dc86cd7a1598a8f44af4d7ae36"

qlora_model = PeftModel.from_pretrained(core_model, ADAPTER_REPO, revision=ADAPTER_REV)
print("Fine-tuned model loaded successfully!")

In [ ]:
TOP_K_TOKENS = 3
from transformers import set_seed

def topk_weighted_price(prompt: str, top_k: int = TOP_K_TOKENS, seed: int = 42) -> float:
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    input_ids = tok.encode(prompt, return_tensors="pt").to(device)
    attn = torch.ones_like(input_ids, device=device)

    with torch.no_grad():
        out = qlora_model(input_ids, attention_mask=attn)
        logits = out.logits[:, -1, :]

    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = probs.topk(top_k)

    prices, weights = [], []
    for i in range(top_k):
        token_str = tok.decode(top_ids[0, i])
        p = top_probs[0, i].item()
        try:
            num = float(token_str)
        except ValueError:
            continue
        if num > 0:
            prices.append(num)
            weights.append(p)

    if not prices: return 0.0
    weight_sum = sum(weights)
    return sum(v * w / weight_sum for v, w in zip(prices, weights))

In [ ]:
example_product = "Apple AirPods Pro 2nd Gen, wireless earbuds with noise cancellation."
predicted_price = topk_weighted_price(example_product)
print(f"Predicted price: ${predicted_price:,.2f}")

In [ ]:
class Scorecard:
    def __init__(self, predictor, data, title="Product Price Estimator", size=50):
        self.predictor = predictor
        self.data = data
        self.size = min(size, len(data))
        self.preds, self.labels, self.abs_err, self.sle, self.zones = [], [], [], [], []

    def _zone(self, err, truth):
        if err < 5 or err/truth < 0.1: return "green"
        if err < 15 or err/truth < 0.3: return "orange"
        return "red"

    def _eval_one(self, i):
        row = self.data[i]
        pred = self.predictor(row["text"])
        truth = row["price"]
        err = abs(pred-truth)
        sle = (math.log(truth+1)-math.log(pred+1))**2
        zone = self._zone(err, truth)
        self.preds.append(pred); self.labels.append(truth)
        self.abs_err.append(err); self.sle.append(sle); self.zones.append(zone)
        print(f"{i+1}: Guess: ${pred:,.2f} | Truth: ${truth:,.2f} | Error: ${err:,.2f} | Zone: {zone}")

    def _plot(self):
        plt.figure(figsize=(12,8))
        mx = max(max(self.labels), max(self.preds))
        plt.plot([0,mx],[0,mx], color='deepskyblue', lw=2, alpha=0.7)
        plt.scatter(self.labels, self.preds, c=self.zones, alpha=0.6)
        plt.xlabel("Ground Truth Price ($)"); plt.ylabel("Predicted Price ($)")
        plt.grid(True); plt.show()

    def run(self):
        for i in tqdm(range(self.size), desc="Evaluating"):
            self._eval_one(i)
        self._plot()
        avg_err = sum(self.abs_err)/self.size
        print(f"Average Dollar Error: ${avg_err:,.2f}")

# Run evaluation
Scorecard(topk_weighted_price, test_split, size=50).run()

## Future Improvements

- **Structured Features Input**: Include product category, brand, color, size, etc., for more accurate predictions
- **Chain-of-Thought Reasoning**: Include reasoning steps in training for interpretable predictions
- **RAG / Retrieval-Augmented Data**: Use historical product prices or similar products for context
- **Confidence Scores**: Add a prediction confidence output
- **Custom Loss**: RMSLE or log-price loss for better numeric alignment
- **Deployment**: Wrap as a FastAPI service or SaaS internal tool for e-commerce pricing